# Day 18 — Syscalls, Visualized

`open().read()` bottoms out in kernel **system calls**. Let's use them directly, and see
why buffer size changes how many times you cross the user/kernel boundary.

In [ ]:
import os

path = '/tmp/lp_day18_demo.txt'
# Raw syscalls -- no Python file object:
fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o644)
os.write(fd, b'hello, kernel')
os.close(fd)

fd = os.open(path, os.O_RDONLY)
print('read back:', os.read(fd, 100))
print('size via lseek:', os.lseek(fd, 0, os.SEEK_END), 'bytes')
os.close(fd)

## Buffer size = number of syscalls

Reading a file in tiny chunks means many `read` syscalls; a big buffer means few. Each
crossing has a cost — this is why buffering exists.

In [ ]:
data = bytes(10000)
fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC); os.write(fd, data); os.close(fd)

for chunk in (1, 64, 4096, 65536):
    fd = os.open(path, os.O_RDONLY); calls = 0
    while True:
        calls += 1
        if not os.read(fd, chunk): break
    os.close(fd)
    print(f'chunk={chunk:6d} bytes -> {calls:5d} read() syscalls')

print('\n(on Linux, run  strace -c python yourscript.py  to count real syscalls)')

## Takeaways

- A syscall is the user-space -> kernel doorway; file I/O, sockets, everything goes through it.
- A file descriptor is just a small integer the kernel hands back.
- Fewer, bigger operations beat many tiny ones — that's what buffering buys you.

**Now build** low_level_write/read/copy + file_size in `homework.py`, then `pytest -q`.